[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/02_dbscan/DBSCAN.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/02_dbscan/DBSCAN.ipynb)

# Notebook 02 — DBSCAN
## Finding Clusters by Density, Ignoring Noise

**Dataset**: Iris flowers
**Question**: *Can we find groups based on how densely packed the flowers are?*
**Type**: Unsupervised — density-based clustering

---

## Section 1 — What Is DBSCAN?

### In plain English

Imagine you are at a crowded concert looking down from above.
You can see clusters of people — dense groups standing together, watching the stage.
In between the clusters there are scattered individuals — people walking to the exits.

**DBSCAN finds clusters by looking for dense regions.
Isolated points in sparse areas are labelled "noise" — they don't belong to any cluster.**

> DBSCAN = **D**ensity-**B**ased **S**patial **C**lustering of **A**pplications with **N**oise

### Key advantages over K-Means

| | K-Means | DBSCAN |
|---|---|---|
| Must specify K | Yes — you decide upfront | **No — K emerges from data** |
| Cluster shape | Spherical only | **Any shape** |
| Handles noise/outliers | No — assigns every point | **Yes — marks noise as -1** |
| Works when | Clusters are round, equal size | Clusters are irregular, or data has outliers |

## Section 2 — How Does It Learn?

### Two parameters control what "dense" means

- **eps** (ε): the radius of the neighbourhood around each point
- **min_samples**: minimum number of points needed in the neighbourhood to form a dense region

### Three types of points

**Core point**: has at least `min_samples` neighbours within radius `eps`
→ It is in a dense region — definitely part of a cluster

**Border point**: within eps of a core point, but does not itself have enough neighbours
→ On the edge of a cluster

**Noise point**: not within eps of any core point
→ Isolated — labelled **-1** (no cluster)

### Cluster expansion

1. Pick an unvisited point. If it is a core point, start a new cluster.
2. Add all points within eps to this cluster (they are "reachable").
3. If any of those are also core points, expand from them too (chain reaction).
4. Continue until no more points are reachable.
5. Mark remaining unvisited points as noise or start new clusters.

## Section 3 — What Does the Data Need?

### Clustering is distance-based — scaling is essential

All clustering algorithms group points by how close they are to each other.
Distance is measured in feature space:
```
distance = sqrt((x1_a − x1_b)² + (x2_a − x2_b)² + ...)
```

If `sepal_length` ranges 4–8 cm and `petal_width` ranges 0.1–2.5 cm,
a 1 cm difference in sepal_length contributes equally to distance as a 1 cm difference in petal_width.
But the sepal feature has 4× the range — it dominates the distance calculation.

**Fix: StandardScaler** — all features end up with mean=0 and std=1.
Now every feature contributes equally to the distance.

### What clustering does NOT need

- No target labels (that is the whole point — we do not have them)
- No train/test split (there is no prediction to evaluate on held-out data)
- Only preparation: encode to numbers, fill missing values, scale

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Load Iris — a classic dataset with 3 natural groups of flowers
df_raw = sns.load_dataset('iris')
FEATURES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
TRUE_LABELS = df_raw['species'].map({'setosa': 0, 'versicolor': 1, 'virginica': 2}).values

X_raw = df_raw[FEATURES].values

# Scale features — clustering algorithms are distance-based; scaling is required
scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

print(f"Iris dataset: {X_raw.shape[0]} flowers, {X_raw.shape[1]} features")
print(f"True groups  : {df_raw['species'].value_counts().to_dict()}")
print()
print("We will PRETEND we do not know the species labels.")
print("The clustering algorithm must discover the 3 groups by itself.")
df_raw.head(6)

## Section 4 — Tune Parameters, Train, and Visualise

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Sensitivity to eps — how the number of clusters and noise changes
eps_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
print(f"{'eps':>6}  {'Clusters':>9}  {'Noise pts':>10}  {'Silhouette':>12}")
print("-" * 44)
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    lbl = db.fit_predict(X)
    n_clusters = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_noise    = (lbl == -1).sum()
    sil = silhouette_score(X, lbl) if n_clusters > 1 and n_clusters < len(X)-1 else float('nan')
    print(f"{eps:>6.1f}  {n_clusters:>9}  {n_noise:>10}  {sil:>12.4f}")

In [ ]:
# Use eps=0.5 — 3 clusters, minimal noise
db = DBSCAN(eps=0.5, min_samples=5)
pred_labels = db.fit_predict(X)

n_clusters = len(set(pred_labels)) - (1 if -1 in pred_labels else 0)
n_noise    = (pred_labels == -1).sum()
sil = silhouette_score(X, pred_labels)

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise}")
print(f"Silhouette     : {sil:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_pred = ['#5C6BC0', '#26A69A', '#FFA726', '#EF5350', '#AB47BC']
colors_true = ['#5C6BC0', '#26A69A', '#FFA726']

# DBSCAN result
unique_labels = sorted(set(pred_labels))
for k in unique_labels:
    mask = pred_labels == k
    if k == -1:
        axes[0].scatter(X[mask,2], X[mask,3], c='black', s=20, marker='x',
                        alpha=0.6, label='Noise (outliers)')
    else:
        axes[0].scatter(X[mask,2], X[mask,3], color=colors_pred[k], s=40, alpha=0.8,
                        label=f'Cluster {k}')
axes[0].set_title(f'DBSCAN (eps=0.5, min_samples=5)\n{n_clusters} clusters, {n_noise} noise points', fontsize=12)
axes[0].set_xlabel('petal_length (scaled)', fontsize=11)
axes[0].set_ylabel('petal_width (scaled)', fontsize=11)
axes[0].legend()

# True labels
for k, name in enumerate(['setosa','versicolor','virginica']):
    mask = TRUE_LABELS == k
    axes[1].scatter(X[mask,2], X[mask,3], color=colors_true[k], s=40, alpha=0.8, label=name)
axes[1].set_title('True Species', fontsize=12)
axes[1].set_xlabel('petal_length (scaled)', fontsize=11)
axes[1].set_ylabel('petal_width (scaled)', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()
print()
print('DBSCAN discovered the groups without needing to be told K=3 in advance.')
print('Black X markers = noise points that do not belong to any cluster.')